<a href="https://colab.research.google.com/github/springboardmentor468a/Projects/blob/ImageSegmentation-AryaaAgarwal/Final_Project_File/VisionExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**==============================================================**
# **WEEK 1: IMAGE MASKING AND AUGUMENTATION**
**==============================================================**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage.io as io
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
import cv2

In [ ]:
def resize_image_and_mask(image, mask, size=(255, 255)):
    """Resize both image and mask to the same size"""
    image_resized = cv2.resize(image, size)
    mask_resized = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)
    return image_resized, mask_resized

In [ ]:
def normalize_image(image):
    """Normalize image pixels to range [0,1]"""
    return image.astype(np.float32) / 255.0

In [ ]:
def augment_image_and_mask(image, mask):
    """Apply augmentation: flip, rotate, stretch"""
    h, w = image.shape[:2]

    # Random horizontal flip
    if np.random.rand() > 0.5:
        image = cv2.flip(image, 1)
        mask = cv2.flip(mask, 1)

    # Random rotation (-20 to 20 degrees)
    angle = np.random.uniform(-20, 20)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    image = cv2.warpAffine(image, M, (w, h))
    mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST)

    # Random stretching (scaling X or Y axis)
    scale_x = np.random.uniform(0.8, 1.2)
    scale_y = np.random.uniform(0.8, 1.2)
    M_stretch = np.array([[scale_x, 0, 0], [0, scale_y, 0]], dtype=np.float32)
    image = cv2.warpAffine(image, M_stretch, (w, h))
    mask = cv2.warpAffine(mask, M_stretch, (w, h), flags=cv2.INTER_NEAREST)

    return image, mask

In [ ]:
annFile = 'annotations/instances_val2017.json'
img_path = 'val2017/'

In [ ]:
coco = COCO(annFile)

In [ ]:
catIds = coco.getCatIds(catNms=['person'])
imgIds = coco.getImgIds(catIds=catIds)

In [ ]:
for i, imgId in enumerate(imgIds[:3]):  # take 3 images for demo
    img_info = coco.loadImgs(imgId)[0]
    I = io.imread(img_path + img_info['file_name'])

    annIds = coco.getAnnIds(imgIds=imgId, catIds=catIds, iscrowd=None)
    anns = coco.loadAnns(annIds)

    # --------------------------
    # Create Mask (person only)
    # --------------------------
    mask = np.zeros((I.shape[0], I.shape[1]), dtype=np.uint8)
    for ann in anns:
        rle = coco.annToRLE(ann)
        m = maskUtils.decode(rle)
        mask = np.maximum(mask, m)

    # Apply mask to image (keep only person region)
    masked_img = I * mask[:, :, np.newaxis]

    # --------------------------
    # Preprocessing + Augmentation
    # --------------------------
    I_resized, mask_resized = resize_image_and_mask(masked_img, mask, (255, 255))
    I_norm = normalize_image(I_resized)

    I_aug, mask_aug = augment_image_and_mask(I_resized, mask_resized)

    # --------------------------
    # Show Results
    # --------------------------
    plt.figure(figsize=(18, 8))

    plt.subplot(1, 4, 1)
    plt.imshow(I)
    plt.title(f"Original - {img_info['file_name']}")

    plt.subplot(1, 4, 2)
    plt.imshow(masked_img)
    plt.title("Masked (person only)")

    plt.subplot(1, 4, 3)
    plt.imshow(I_norm)
    plt.title("Preprocessed (Resized + Normalized)")

    plt.subplot(1, 4, 4)
    plt.imshow(I_aug)
    plt.title("Augmented (Flip/Rotate/Stretch)")

    plt.show()

**=============================================================**
# **WEEK 2: IMAGE SEGMENTAION USING CATEGORY BEAR**
**=============================================================**

In [ ]:
import os, random
import albumentations as Album

In [ ]:
bear_catId = coco.getCatIds(catNms=["bear"])[0]
bear_imgIds = coco.getImgIds(catIds=[bear_catId])
random.shuffle(bear_imgIds)

In [ ]:
sample_imgIds = bear_imgIds[:5]

In [ ]:
transform = Album.Compose([ Album.Resize(255, 255),
                           Album.RandomRotate90(),
                           Album.HorizontalFlip(p=0.5),
                           Album.VerticalFlip(p=0.5),
                           Album.ShiftScaleRotate(shift_limit=0.2, scale_limit=0.3, rotate_limit=60, p=0.7),
                           Album.RandomBrightnessContrast(p=0.5),
                           Album.HueSaturationValue(p=0.5),
                           Album.GaussianBlur(blur_limit=(3, 7), p=0.4),
                           Album.RandomGamma(p=0.4),
                           Album.CoarseDropout(max_holes=8, max_height=40, max_width=40, p=0.3)
                          ])

In [ ]:
for idx, imgId in enumerate(sample_imgIds, 1):
    imgInfo = coco.loadImgs(imgId)[0]
    # Load image
    imgPath = os.path.join(imgDir, imgInfo['file_name'])
    image = cv2.cvtColor(cv2.imread(imgPath), cv2.COLOR_BGR2RGB)
    # Load bear annotations
    annIds = coco.getAnnIds(imgIds=imgInfo['id'], catIds=[bear_catId])
    anns = coco.loadAnns(annIds)
    # Build binary mask(bear=1, background=0)
    mask = np.zeros((imgInfo['height'], imgInfo['width']), dtype=np.uint8)
    for ann in anns:
        mask = np.maximum(mask, coco.annToMask(ann).astype(np.uint8))
        # Resize + normalize image
        image_resized = cv2.resize(image, (255, 255)).astype(np.float32) / 255.0
        mask_resized = cv2.resize(mask, (255, 255), interpolation=cv2.INTER_NEAREST)
        # Binary mask (bear=white, background=black)
        binary_mask = (mask_resized * 255).astype(np.uint8)
        # ============================
        # Apply augmentation
        # ============================
        augmented = transform(image=image, mask=mask)
        aug_img, aug_mask = augmented["image"], augmented["mask"]
        aug_img_norm = aug_img.astype(np.float32) / 255.0
        aug_binary_mask = (aug_mask * 255).astype(np.uint8)
        # ============================
        # Show results
        # ============================
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(image_resized)
        ax[0].set_title("Original Bear (Resized + Normalized)")
        ax[0].axis("off")
        ax[1].imshow(aug_img_norm)
        ax[1].set_title("Augmented Bear Image")
        ax[1].axis("off")
        ax[2].imshow(aug_binary_mask, cmap="gray")
        ax[2].set_title("Binary Mask (Bear=White, BG=Black)")
        ax[2].axis("off")
        plt.show()

**================================================================**
# **WEEK 3: Training**
**================================================================**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models.segmentation as segmentation
import torchvision.transforms as T

In [ ]:
bear_catId = coco.getCatIds(catNms=["bear"])[0]

In [ ]:
bear_imgIds = coco.getImgIds(catIds=[bear_catId])
random.shuffle(bear_imgIds)

In [ ]:
sample_imgIds = bear_imgIds[:5]

In [ ]:
class CocoBearDataset(Dataset):
    def __init__(self, coco, imgDir, imgIds, catId, transform=None):
        self.coco = coco
        self.imgDir = imgDir
        self.imgIds = imgIds
        self.catId = catId
        self.transform = transform

    def __len__(self):
        return len(self.imgIds)

    def __getitem__(self, idx):
        imgId = self.imgIds[idx]
        imgInfo = self.coco.loadImgs(imgId)[0]

        # Load image
        imgPath = os.path.join(self.imgDir, imgInfo['file_name'])
        image = cv2.cvtColor(cv2.imread(imgPath), cv2.COLOR_BGR2RGB)

        # Load mask (bear=1, background=0)
        annIds = self.coco.getAnnIds(imgIds=imgInfo['id'], catIds=[self.catId])
        anns = self.coco.loadAnns(annIds)
        mask = np.zeros((imgInfo['height'], imgInfo['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann).astype(np.uint8))

        # Resize
        image = cv2.resize(image, (255, 255))
        mask = cv2.resize(mask, (255, 255), interpolation=cv2.INTER_NEAREST)

        # Albumentations augmentation
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        # Convert to tensor
        image = T.ToTensor()(image)  # [C,H,W] float32
        mask = torch.tensor(mask, dtype=torch.long)  # [H,W] long

        return image, mask

In [ ]:
split = int(0.8 * len(bear_imgIds))
train_ids, val_ids = bear_imgIds[:split], bear_imgIds[split:]

In [ ]:
train_transform = Album.Compose([
    Album.Resize(255, 255),
    Album.RandomRotate90(),
    Album.HorizontalFlip(p=0.5),
    Album.VerticalFlip(p=0.5),
    Album.ShiftScaleRotate(shift_limit=0.2, scale_limit=0.3, rotate_limit=60, p=0.7),
    Album.RandomBrightnessContrast(p=0.5),
    Album.HueSaturationValue(p=0.5),
    Album.GaussianBlur(blur_limit=(3, 7), p=0.4),
    Album.RandomGamma(p=0.4),
    Album.CoarseDropout(max_holes=8, max_height=40, max_width=40, p=0.3)
])


In [ ]:
val_transform = Album.Compose([
    Album.Resize(255, 255)
])

In [ ]:
train_dataset = CocoBearDataset(coco, imgDir, train_ids, bear_catId, transform=train_transform)
val_dataset   = CocoBearDataset(coco, imgDir, val_ids,   bear_catId, transform=val_transform)


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define model first
model = segmentation.deeplabv3_resnet50(pretrained=True)
num_classes = 2
model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
model = model.to(device)

# Then define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)["out"]
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)["out"]
        preds = torch.argmax(outputs, dim=1).cpu()

        # Convert image back to numpy
        orig = images[0].cpu().permute(1, 2, 0).numpy()
        orig = (orig * 255).astype(np.uint8)

        gt_mask = masks[0].cpu().numpy()
        pred_mask = preds[0].numpy()

        # Apply mask to image (background = black)
        gt_segmented = orig.copy()
        gt_segmented[gt_mask == 0] = [0, 0, 0]

        pred_segmented = orig.copy()
        pred_segmented[pred_mask == 0] = [0, 0, 0]

        # Plot
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(orig)
        ax[0].set_title("Original Image")

        ax[1].imshow(gt_segmented)
        ax[1].set_title("Ground Truth Segmentation")

        ax[2].imshow(pred_segmented)
        ax[2].set_title("Predicted Segmentation")

        plt.show()
        break


In [ ]:
def compute_metrics(outputs, masks, num_classes=2):
    """
    outputs: [N,H,W] predicted class ids
    masks:   [N,H,W] ground truth class ids
    """
    outputs = outputs.cpu().numpy()
    masks = masks.cpu().numpy()

    total_correct = 0
    total_pixels = 0
    iou_list = []

    for cls in range(num_classes):
        intersection = np.logical_and(outputs == cls, masks == cls).sum()
        union = np.logical_or(outputs == cls, masks == cls).sum()
        if union > 0:
            iou_list.append(intersection / union)

    total_correct += (outputs == masks).sum()
    total_pixels  += np.prod(masks.shape)

    overall_acc = total_correct / total_pixels
    mean_iou = np.mean(iou_list)

    return overall_acc, mean_iou

In [ ]:
model.eval()
total_acc_sum = 0
miou_sum = 0
count = 0

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)["out"]
        preds = torch.argmax(outputs, dim=1)

        acc, miou = compute_metrics(preds, masks, num_classes=2)
        total_acc_sum += acc
        miou_sum += miou
        count += 1

pixel_acc = total_acc_sum / count
mean_iou  = miou_sum / count

In [ ]:
print(f"Validation Pixel Accuracy: {pixel_acc:.4f}")
print(f"Validation Mean IoU: {mean_iou:.4f}")

**=================================================================**
# **WEEK 4: HYPERPARAMETERS**
**=================================================================**

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import numpy as np

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
annFile = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/annotations/instances_val2017.json"
root = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/val2017"

from torchvision.datasets import CocoDetection
dataset = CocoDetection(root=root, annFile=annFile)


In [ ]:
from torchvision.datasets import CocoDetection

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths (update to your Google Drive structure)
train_root = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/val2017"  # only val2017 exists in your Drive
train_ann = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/annotations/instances_val2017.json"

val_root = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/val2017"
val_ann = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/annotations/instances_val2017.json"

!pip install pycocotools -q

# Custom dataset wrapper
class CocoSegmentation(torch.utils.data.Dataset):
    def __init__(self, root, annFile, transforms=None):
        self.dataset = CocoDetection(root, annFile)
        self.transforms = transforms

    def __getitem__(self, idx):
        img, target = self.dataset[idx]
        img = np.array(img)

        # Create binary mask (object vs background)
        mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        for obj in target:
            rle = self.dataset.coco.annToMask(obj)
            mask = np.maximum(mask, rle)

        # Convert mask to float32 before applying transforms
        mask = mask.astype(np.float32)

        if self.transforms:
            augmented = self.transforms(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]

        return img, mask.long()

    def __len__(self):
        return len(self.dataset)

# Augmentations
train_transform = A.Compose([
    A.Resize(height=256, width=256), # Resize first
    A.RandomCrop(height=256, width=256), # Then random crop
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


# Use val2017 for both train/val (since train2017 is not uploaded)
train_dataset = CocoSegmentation(
    root=train_root,
    annFile=train_ann,
    transforms=train_transform
)

val_dataset = CocoSegmentation(
    root=val_root,
    annFile=val_ann,
    transforms=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("Train images:", len(train_dataset), "Val images:", len(val_dataset))

In [ ]:
import torchvision.models.segmentation as models

# Model
model = models.deeplabv3_resnet50(pretrained=True)
model.classifier[4] = torch.nn.Conv2d(256, 2, kernel_size=1)  # 2 classes: background + object

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def dice_coefficient(pred, target, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)


In [ ]:
num_epochs = 20  # try small first
best_val_dice = 0.0
save_path = "/content/drive/MyDrive/Colab Notebooks/VisionExtraction/best_model.pth"

for epoch in range(num_epochs):
    model.train()
    train_loss, train_dice = 0, 0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_dice += dice_coefficient(outputs, masks).item()

    # Validation
    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)['out']
            loss = criterion(outputs, masks)

            val_loss += loss.item()
            val_dice += dice_coefficient(outputs, masks).item()

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss/len(train_loader):.4f}, Train Dice: {train_dice/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}, Val Dice: {val_dice/len(val_loader):.4f}")

    # Save best model
    if val_dice/len(val_loader) > best_val_dice:
        best_val_dice = val_dice/len(val_loader)
        torch.save(model.state_dict(), save_path)
        print("✅ Model saved!")


In [ ]:
import matplotlib.pyplot as plt

model.eval()
images, masks = next(iter(val_loader))
images, masks = images.to(device), masks.to(device)

with torch.no_grad():
    outputs = model(images)['out']
preds = torch.argmax(outputs, dim=1).cpu().numpy()

plt.figure(figsize=(12,6))
for i in range(3):
    plt.subplot(3,3,i*3+1)
    plt.imshow(images[i].cpu().permute(1,2,0))
    plt.axis("off"); plt.title("Image")

    plt.subplot(3,3,i*3+2)
    plt.imshow(masks[i].cpu(), cmap="gray")
    plt.axis("off"); plt.title("Ground Truth")

    plt.subplot(3,3,i*3+3)
    plt.imshow(preds[i], cmap="gray")
    plt.axis("off"); plt.title("Prediction")

plt.show()


**================================================================**
# **WEEK 5: FINAL TRAINED MODEL**
**================================================================**

<a href="https://colab.research.google.com/github/springboardmentor468a/Projects/blob/ImageSegmentation-AryaaAgarwal/week5_FinalTrainedMode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
from tqdm import tqdm

In [ ]:
class CocoSegmentation(torch.utils.data.Dataset):
    def __init__(self, root, annFile, transforms=None):
        self.dataset = CocoDetection(root, annFile)
        self.transforms = transforms

    def __getitem__(self, idx):
        img, target = self.dataset[idx]
        img = np.array(img)

        # Binary mask: object vs background
        mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        for obj in target:
            rle = self.dataset.coco.annToMask(obj)
            mask = np.maximum(mask, rle)

        mask = mask.astype(np.float32)

        if self.transforms:
            augmented = self.transforms(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]

        return img, mask.long()

    def __len__(self):
        return len(self.dataset)

In [ ]:
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
train_dataset = CocoSegmentation(train_root, train_ann, transforms=train_transform)
val_dataset   = CocoSegmentation(val_root, val_ann, transforms=val_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False,
                          num_workers=4, pin_memory=True)

print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))

In [ ]:
model = models.deeplabv3_resnet50(pretrained=True)
model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)  # 2 classes
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision

def dice_coefficient(pred, target, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

In [ ]:
num_epochs = 30
best_val_dice = 0.0
save_path = "/content/drive/MyDrive/VisionExtraction/Projects/best_model.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    train_loss, train_dice = 0, 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -------------------------------
    # Training
    # -------------------------------
    for images, masks in tqdm(train_loader, desc="Training", unit="batch", ncols=100):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)['out']
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_dice += dice_coefficient(outputs, masks).item()

    # -------------------------------
    # Validation
    # -------------------------------
    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc="Validation", unit="batch", ncols=100):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)['out']
            loss = criterion(outputs, masks)

            val_loss += loss.item()
            val_dice += dice_coefficient(outputs, masks).item()

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1} completed in {epoch_time:.2f}s")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Dice: {train_dice/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}, Val Dice: {val_dice/len(val_loader):.4f}")

    # Save best model
    if val_dice/len(val_loader) > best_val_dice:
        best_val_dice = val_dice/len(val_loader)
        torch.save(model.state_dict(), save_path)
        print("✅ Model saved!")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Number of images you want to display
num_to_show = 10

# Collect images and masks from multiple batches if needed
images_list = []
masks_list = []

collected = 0
val_loader_iter = iter(val_loader)

while collected < num_to_show:
    try:
        imgs, msks = next(val_loader_iter)
    except StopIteration:
        break  # No more data in val_loader
    images_list.append(imgs)
    masks_list.append(msks)
    collected += imgs.size(0)

# Concatenate and move to device
images = torch.cat(images_list, dim=0).to(device)
masks = torch.cat(masks_list, dim=0).to(device)

# Run model
model.eval()
with torch.no_grad():
    outputs = model(images)['out']
preds = torch.argmax(outputs, dim=1)

# Only keep up to num_to_show images
num_samples = min(num_to_show, len(images))

# Denormalize function (adjust if your dataset uses different mean/std)
def denormalize(img_tensor, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]):
    img = img_tensor.clone().cpu()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return np.clip(img.permute(1,2,0).numpy(), 0, 1)

# Plotting
plt.figure(figsize=(12, 4 * num_samples))

for i in range(num_samples):
    pred_mask = (preds[i] == 1).cpu().numpy().astype(np.uint8)
    gt_mask   = (masks[i] == 1).cpu().numpy().astype(np.uint8)

    img_np = denormalize(images[i])
    img_masked_pred = img_np * pred_mask[:, :, None]
    img_masked_gt   = img_np * gt_mask[:, :, None]

    # Original Image
    plt.subplot(num_samples, 3, i*3+1)
    plt.imshow(img_np)
    plt.axis("off")
    plt.title("Original Image")
    plt.gca().set_facecolor('black')

    # Ground truth
    plt.subplot(num_samples, 3, i*3+2)
    plt.imshow(img_masked_gt)
    plt.axis("off")
    plt.title("Ground Truth")
    plt.gca().set_facecolor('black')

    # Prediction
    plt.subplot(num_samples, 3, i*3+3)
    plt.imshow(img_masked_pred)
    plt.axis("off")
    plt.title("Predicted")
    plt.gca().set_facecolor('black')

plt.tight_layout()
plt.show()


**================================================================**
# **WEEK 6: TESTING ON NEW IMAGES ON INTERNET**
**================================================================**

In [ ]:
import cv2

def denormalize(img_tensor, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]):
    img = img_tensor.clone().cpu()
    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)
    return np.clip(img.permute(1,2,0).numpy(), 0, 1)

def run_inference_on_image(image_path):
    # Load and preprocess
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Resize image to match model input ---
    img_resized = cv2.resize(img, (256, 256))

    transformed = val_transform(image=img, mask=np.zeros(img.shape[:2], dtype=np.uint8))
    img_tensor = transformed["image"].unsqueeze(0).to(device)

    # Inference
    with torch.no_grad():
        output = model(img_tensor)["out"]
    pred = torch.argmax(output, dim=1)[0].cpu().numpy()

    # Create black background and keep only object
    object_only = np.zeros_like(img_resized)
    object_only[pred == 1] = img_resized[pred == 1]

    # Visualization
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.imshow(img_resized)
    plt.axis('off')
    plt.title("Original (Resized)")

    plt.subplot(1,2,2)
    plt.imshow(object_only)
    plt.axis('off')
    plt.title("Predicted Object")
    plt.show()


In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

# Allow multiple image uploads
upload_button = widgets.FileUpload(accept='image/*', multiple=True)

def on_upload_change(change):
    if upload_button.value:
        for file_info in upload_button.value.values():
            file_name = file_info['metadata']['name']
            with open(file_name, "wb") as f:
                f.write(file_info['content'])
            print(f"📷 Running inference on {file_name} ...")
            run_inference_on_image(file_name)

upload_button.observe(on_upload_change, names='value')
display(upload_button)
